# Topological Analysis of Financial Market Stress

## Project Overview

This project investigates whether **Topological Data Analysis (TDA)** can reveal structural changes in financial markets that are not fully captured by classical measures such as volatility and correlation.

The main idea is to represent the returns of a collection of stocks over a rolling time window as a high-dimensional point cloud. Persistent homology is then used to study the geometric and topological structure of these point clouds over time.

The central research question is:

> **Can persistent homology detect changes in market structure that differ from conventional measures of financial market stress?**

---

## Data

The analysis uses historical daily stock-price data downloaded with `yfinance`.

For each stock, daily percentage returns are computed. A rolling window of 100 trading days is then used to construct point clouds.

For a market consisting of $N$ stocks, each trading day is represented by a vector

$$(r_1, r_2, \ldots, r_N),$$

where $r_i$ is the return of stock $i$ on that day.

A 100-day window therefore produces a point cloud consisting of 100 points in $\mathbb{R}^N$.

---

## Topological Data Analysis

For each rolling point cloud, persistent homology is computed using a Vietoris–Rips filtration.

The analysis currently considers:

- $H_1$: one-dimensional topological features, corresponding to loops
- $H_2$: two-dimensional topological features, corresponding to higher-dimensional cavities

For every topological feature, its persistence is defined as

$$
p_i = d_i - b_i,
$$

where $b_i$ and $d_i$ denote its birth and death values.

To summarize a persistence diagram by a single number, the project uses the \(L^2\)-norm of the persistence values:

$$
S_{\mathrm{raw}}
=
\sqrt{
\sum_i p_i^2
}.
$$

This gives a **raw TDA stress measure**.

---

## Normalized TDA Stress

Raw persistence depends partly on the overall geometric scale of the point cloud.

During periods of high volatility, stock returns become larger and the point cloud expands. Persistence values can therefore increase even if the underlying topological shape remains similar.

To distinguish geometric scale from topological structure, a second measure normalizes the TDA stress by a characteristic scale of the point cloud:

$$
S_{\mathrm{normalized}}
=
\frac{S_{\mathrm{raw}}}
{\operatorname{median}_{i<j}\|x_i-x_j\|_2}.
$$

This produces two conceptually different quantities:

- **Raw TDA stress:** contains both market scale and topological structure
- **Normalized TDA stress:** focuses more strongly on relative topological structure

Both $H_1$ and $H_2$ versions are calculated.

---

## Classical Market Measures

The TDA measures are compared with two conventional financial indicators.

### Average Stock Volatility

For each stock $i$, let

$$
r_{1,i},\ldots,r_{T,i}
$$

denote its returns within a rolling window of \(T=100\) trading days.

First, the average return of stock $i$ is computed:

$$
\bar r_i
=
\frac{1}{T}
\sum_{t=1}^{T} r_{t,i}.
$$

Its sample variance is then

$$
s_i^2
=
\frac{1}{T-1}
\sum_{t=1}^{T}
(r_{t,i}-\bar r_i)^2,
$$

and the corresponding volatility is the sample standard deviation

$$
\sigma_i = \sqrt{s_i^2}.
$$

Finally, the volatilities of all $N$ stocks are averaged:

$$
V
=
\frac{1}{N}
\sum_{i=1}^{N}
\sigma_i.
$$

Thus, $V$ measures the typical magnitude of return fluctuations of an individual stock during the 100-day window.

### Average Correlation

For each pair of stocks $i$ and $j$, the Pearson correlation of their returns within the 100-day window is calculated.

Let $r_{t,i}$ and $r_{t,j}$ denote the returns of stocks $i$ and $j$ on day $t$. Their Pearson correlation is

$$
\rho_{ij}
=
\frac{
\sum_{t=1}^{T}
(r_{t,i}-\bar r_i)(r_{t,j}-\bar r_j)
}{
\sqrt{
\sum_{t=1}^{T}(r_{t,i}-\bar r_i)^2
}
\sqrt{
\sum_{t=1}^{T}(r_{t,j}-\bar r_j)^2
}
}.
$$

The value $\rho_{ij}$ lies between $-1$ and $1$. Positive values indicate that the two stocks tend to move in the same direction, while negative values indicate movement in opposite directions.

Finally, the correlations of all distinct pairs of stocks are averaged:

$$
\bar{\rho}
=
\frac{2}{N(N-1)}
\sum_{i<j}
\rho_{ij}.
$$

Thus, $\bar{\rho}$ measures the overall degree of co-movement between stocks in the market. High average correlation indicates that many stocks are moving together, which is often associated with periods of market stress.


## Comparison of the Measures

The different measures have different units and numerical scales. To make their time series visually comparable, each measure is standardized.

For a time series $x_1,\ldots,x_n$, let

$$
\mu_x
=
\frac{1}{n}
\sum_{t=1}^{n} x_t
$$

denote its mean and let $\sigma_x$ denote its standard deviation. Each observation is transformed into a z-score

$$
z_t
=
\frac{x_t-\mu_x}{\sigma_x}.
$$

Subtracting $\mu_x$ centers the time series around zero, while dividing by $\sigma_x$ rescales it so that one unit corresponds to one standard deviation.

This makes it possible to compare the relative movements and unusually high or low values of measures whose original scales are very different.

Standardization is used only for visualization. The original values are retained for statistical analysis.

The interactive plot below compares:

- H1 raw stress
- H1 normalized stress
- H2 raw stress
- H2 normalized stress
- average stock volatility
- average stock correlation

The curves can be individually shown or hidden to investigate their behavior during different market periods.


## Correlation Analysis

Pearson and Spearman correlations are used to compare the TDA-based measures with the classical market indicators.

The **Pearson correlation** measures the strength of a linear relationship between two variables. The **Spearman correlation** instead compares the ranks of the observations and therefore measures more general monotonic relationships while being less sensitive to extreme values.

The resulting correlations are:

| TDA measure | Pearson volatility | Spearman volatility | Pearson avg. correlation | Spearman avg. correlation |
|---|---:|---:|---:|---:|
| H1 raw | 0.609 | 0.566 | 0.081 | 0.123 |
| H1 normalized | -0.252 | -0.087 | -0.207 | -0.143 |
| H2 raw | 0.165 | 0.149 | -0.085 | -0.038 |
| H2 normalized | -0.175 | -0.107 | -0.163 | -0.119 |

The clearest relationship occurs between **raw H1 stress and volatility**, with a Pearson correlation of approximately $0.61$ and a Spearman correlation of approximately $0.57$. Thus, larger H1 persistence tends to occur during periods of higher volatility. This supports the idea that raw persistence is strongly influenced by the geometric scale of the return point cloud.

After normalization, this positive relationship disappears. The Pearson correlation between normalized H1 stress and volatility is approximately $-0.25$, while the Spearman correlation is only about $-0.09$. This suggests that the negative relationship is weak and not a consistent monotonic pattern across the entire data set.

The H2 measures show only weak correlations with volatility, with absolute values below $0.18$. Similarly, none of the TDA measures shows a strong relationship with average stock correlation. The largest absolute value in this comparison is approximately $0.21$ for normalized H1 stress.

Overall, the correlation analysis does not reveal a strong new market-stress indicator beyond the classical measures. The main result is instead methodological: **raw H1 persistence is substantially related to volatility, while normalization removes most of this scale dependence**. The normalized TDA measures appear to contain information that is largely different from volatility and average correlation, but the present analysis does not establish that this information is itself a useful indicator of market stress.

In [1]:
import computing_table as ct

[*********************100%***********************]  24 of 24 completed


Processing point cloud 5364/5364 (100.0%)

In [2]:
analysis_table = ct.analysis_table

In [3]:
def standardize(series):
    return (series - series.mean()) / series.std()

In [4]:
h1_raw_scaled = standardize(analysis_table["H1_raw_stress"])
h1_normalized_scaled = standardize(analysis_table["H1_normalized_stress"])
h2_raw_scaled = standardize(analysis_table["H2_raw_stress"])
h2_normalized_scaled = standardize(analysis_table["H2_normalized_stress"])
volatility_scaled = standardize(analysis_table["volatility"])
correlation_scaled = standardize(analysis_table["average_correlation"])

In [8]:
import plotly.graph_objects as go

In [33]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=analysis_table.index, y=h1_raw_scaled, mode="lines", name="H1 raw stress"))
fig.add_trace(go.Scatter(x=analysis_table.index, y=h1_normalized_scaled, mode="lines", name="H1 normalized stress"))
fig.add_trace(go.Scatter(x=analysis_table.index, y=h2_raw_scaled, mode="lines", name="H2 raw stress"))
fig.add_trace(go.Scatter(x=analysis_table.index, y=h2_normalized_scaled, mode="lines", name="H2 normalized stress"))
fig.add_trace(go.Scatter(x=analysis_table.index, y=volatility_scaled, mode="lines", name="Average stock volatility"))
fig.add_trace(go.Scatter(x=analysis_table.index, y=correlation_scaled, mode="lines", name="Average correlation"))

fig.add_hline(y=0, line_width=1)

fig.update_layout(
    title="TDA Stress and Classical Market Stress Measures",
    xaxis_title="Date",
    yaxis_title="Standardized value",
    width=950,
    height=600
)

fig.show()

In [35]:
print("Pearson - Volatility")
print("H1 raw:", analysis_table["H1_raw_stress"].corr(analysis_table["volatility"]))
print("H1 normalized:", analysis_table["H1_normalized_stress"].corr(analysis_table["volatility"]))
print("H2 raw:", analysis_table["H2_raw_stress"].corr(analysis_table["volatility"]))
print("H2 normalized:", analysis_table["H2_normalized_stress"].corr(analysis_table["volatility"]))

print("\nPearson - Average Correlation")
print("H1 raw:", analysis_table["H1_raw_stress"].corr(analysis_table["average_correlation"]))
print("H1 normalized:", analysis_table["H1_normalized_stress"].corr(analysis_table["average_correlation"]))
print("H2 raw:", analysis_table["H2_raw_stress"].corr(analysis_table["average_correlation"]))
print("H2 normalized:", analysis_table["H2_normalized_stress"].corr(analysis_table["average_correlation"]))

print("\nSpearman - Volatility")
print("H1 raw:", analysis_table["H1_raw_stress"].corr(analysis_table["volatility"], method="spearman"))
print("H1 normalized:", analysis_table["H1_normalized_stress"].corr(analysis_table["volatility"], method="spearman"))
print("H2 raw:", analysis_table["H2_raw_stress"].corr(analysis_table["volatility"], method="spearman"))
print("H2 normalized:", analysis_table["H2_normalized_stress"].corr(analysis_table["volatility"], method="spearman"))

print("\nSpearman - Average Correlation")
print("H1 raw:", analysis_table["H1_raw_stress"].corr(analysis_table["average_correlation"], method="spearman"))
print("H1 normalized:", analysis_table["H1_normalized_stress"].corr(analysis_table["average_correlation"], method="spearman"))
print("H2 raw:", analysis_table["H2_raw_stress"].corr(analysis_table["average_correlation"], method="spearman"))
print("H2 normalized:", analysis_table["H2_normalized_stress"].corr(analysis_table["average_correlation"], method="spearman"))

Pearson - Volatility
H1 raw: 0.6087602619909018
H1 normalized: -0.25220504038271313
H2 raw: 0.16450018966339372
H2 normalized: -0.1746393425778525

Pearson - Average Correlation
H1 raw: 0.08054982791877206
H1 normalized: -0.2068306569234007
H2 raw: -0.08487513608178128
H2 normalized: -0.16297886111389998

Spearman - Volatility
H1 raw: 0.5655708025866376
H1 normalized: -0.08691175873734362
H2 raw: 0.14931360838687505
H2 normalized: -0.10724664380423841

Spearman - Average Correlation
H1 raw: 0.12339733309749158
H1 normalized: -0.14275197373542536
H2 raw: -0.03778141408306832
H2 normalized: -0.11919978437617268
